In [69]:
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
import torch.nn as nn
from sklearn.datasets import make_classification
import torch

In [70]:
torch.manual_seed(42)

In [71]:
df = pd.read_csv("/fmnist_small.csv")
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0


In [72]:
X = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [73]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = 0.2, random_state=42)

In [74]:
X_train = X_train/255.0
X_test = X_test/255.0

In [75]:
from torch.utils.data import Dataset, DataLoader

In [76]:
#create a CustomDataset class
class CustomDataset(Dataset):
  def __init__(self,features,labels):
    self.features = torch.tensor(features, dtype = torch.float32)
    self.labels = torch.tensor(labels, dtype = torch.long)
  def __len__(self):
    return self.features.shape[0]
  def __getitem__(self,index):
    return self.features[index],self.labels[index]

In [77]:
#create train_dataset object
train_dataset = CustomDataset(X_train,y_train)

In [78]:
#create test_dataset object
test_dataset = CustomDataset(X_test,y_test)


In [79]:
#create train and test loader
train_loader = DataLoader(train_dataset, batch_size = 32 ,shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 32, shuffle =False) # Changed train_dataset to test_dataset

In [80]:
from torch.nn.modules.batchnorm import BatchNorm1d
from torch.nn.modules.activation import ReLU
#define NN class
class MyNN(nn.Module):
  def __init__(self,num_features):

    super().__init__()
    self.model = nn.Sequential(
        nn.Linear(num_features,128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(128,64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(64,10),

    )
    ##we dont need to explicitly define softmax here as in pytorch in cross entropy loss has softmax implemented internally
  def forward(self,X):
    return self.model(X)

In [81]:
# set learning rate and epochs
learning_rate = 0.01
epochs = 100

In [82]:
#instatiate the model
import torch.optim as optim
mynn = MyNN(X_train.shape[1])


#loss function
loss_function = nn.CrossEntropyLoss()
#optimizer

optimizer = optim.SGD(mynn.parameters(), lr=learning_rate,weight_decay = 1e-4)


In [83]:
#training loop:
for epoch in range(epochs):
  total_epoch_loss= 0
  for batch_train , batch_label in train_loader:
    #forward pass
    y_pred = mynn(batch_train)


    #calculate loss
    loss = loss_function(y_pred ,batch_label)
    #back pass
    optimizer.zero_grad()
    loss.backward()


    #upgrade grades
    optimizer.step()
    total_epoch_loss += loss.item()
    avg_loss = total_epoch_loss / len(train_loader)

  print(f'epoch {epoch+1} : {avg_loss}')




epoch 1 : 1.5089164952437082
epoch 2 : 1.045105258623759
epoch 3 : 0.8848111383120218
epoch 4 : 0.7874258255958557
epoch 5 : 0.7123881542682647
epoch 6 : 0.6698581208785375
epoch 7 : 0.6317143032948176
epoch 8 : 0.6078928540150325
epoch 9 : 0.5811879976590475
epoch 10 : 0.5623901559909185
epoch 11 : 0.5363458176453908
epoch 12 : 0.5111375018954277
epoch 13 : 0.514677528043588
epoch 14 : 0.4853396463394165
epoch 15 : 0.47798547595739366
epoch 16 : 0.45272126684586206
epoch 17 : 0.4479185409347216
epoch 18 : 0.4534148833155632
epoch 19 : 0.42421287059783935
epoch 20 : 0.42342266778151194
epoch 21 : 0.39803547620773316
epoch 22 : 0.4085314972201983
epoch 23 : 0.37866468608379367
epoch 24 : 0.37067702730496727
epoch 25 : 0.37389760434627534
epoch 26 : 0.3639563559492429
epoch 27 : 0.3546405978004138
epoch 28 : 0.3467315165201823
epoch 29 : 0.33412749965985616
epoch 30 : 0.32830112730463346
epoch 31 : 0.34196864858269693
epoch 32 : 0.30747159600257873
epoch 33 : 0.3198153485854467
epoch 34 

In [84]:
#set model to eval mode
mynn.eval()
#there are certain features which are different in eval and train, dropout, mean-std calculation

MyNN(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [85]:
#evaluation code
total = 0
correct = 0

with torch.no_grad():

  for batch_features, batch_labels in test_loader:

    outputs = mynn(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

0.8316666666666667


In [86]:
#evaluation code
total = 0
correct = 0

with torch.no_grad():

  for batch_features, batch_labels in train_loader:

    outputs = mynn(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

0.995
